# Pipeline 3 - Geracao de embeddings ATUALIZADA

Versao atualizada: entrada do SPECTER usa somente **title + abstract_reduzido**, sem `id` e sem categoria/subgenero como feature.


## 0. Ative a GPU (recomendado)
No Colab: **Ambiente de execucao  Alterar o tipo de ambiente de execucao  GPU (T4)**. Acelera muito o `encode`.

In [ ]:
!pip -q install sentence-transformers

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Configuracao

In [ ]:
MODELO = 'sentence-transformers/allenai-specter'   # BERT p/ artigos cientificos
USAR_TITULO = True                                # True -> 'title [SEP] abstract_reduzido'
CAMPO_TEXTO = 'abstract_reduzido'                 # campo do abstract enxuto
INCLUIR_ID = False                                # nunca usar id como entrada do embedding
INCLUIR_CATEGORIA = False                         # nunca usar categoria/subgenero como entrada do embedding
BATCH_SIZE = 32

print('Modelo:', MODELO)
print('Entrada: title + abstract_reduzido | sem id | sem assigned_category')


## 2. Montar o Drive e carregar a Pipeline 2 (JSON Lines)

In [ ]:
from google.colab import drive
import os, pandas as pd

drive.mount('/content/drive')

# If needed, put the exact folder path here after checking the Colab file browser.
# Example: BASE = '/content/drive/MyDrive/nome-da-pasta'
BASE = None
NOME_ENTRADA = 'arxiv_amostra_1500_abstracts_limitados.json'


def shallow_find(root, filename, max_depth=3):
    root = os.path.abspath(root)
    root_depth = root.rstrip(os.sep).count(os.sep)
    for current, dirs, files in os.walk(root):
        depth = current.rstrip(os.sep).count(os.sep) - root_depth
        if depth >= max_depth:
            dirs[:] = []
        if filename in files:
            return os.path.join(current, filename)
    return None

candidatos = []
if BASE:
    candidatos.extend([
        os.path.join(BASE, NOME_ENTRADA),
        os.path.join(BASE, 'pipelines', NOME_ENTRADA),
    ])

# Common locations. This is fast and does not scan the whole Drive recursively.
candidatos.extend([
    os.path.join('/content/drive/MyDrive', NOME_ENTRADA),
    os.path.join('/content/drive/MyDrive', 'pipelines', NOME_ENTRADA),
    os.path.join('/content/drive/MyDrive', 'projetoIA-EquipeLoremIpsum', NOME_ENTRADA),
    os.path.join('/content/drive/MyDrive', 'projetoIA-EquipeLoremIpsum', 'pipelines', NOME_ENTRADA),
    os.path.join('/content/drive/MyDrive', 'ProjetoIA-EquipeLoremIpsum', NOME_ENTRADA),
    os.path.join('/content/drive/MyDrive', 'ProjetoIA-EquipeLoremIpsum', 'pipelines', NOME_ENTRADA),
])

CAMINHO_IN = next((p for p in candidatos if os.path.exists(p)), None)

if CAMINHO_IN is None:
    print('File not found in direct paths. Starting shallow search in MyDrive, max_depth=3...')
    CAMINHO_IN = shallow_find('/content/drive/MyDrive', NOME_ENTRADA, max_depth=3)

if CAMINHO_IN is None:
    print('Required file not found:', NOME_ENTRADA)
    print('\nDirect paths tested:')
    for p in candidatos:
        print(' -', p)

    print('\nTop-level items in /content/drive/MyDrive:')
    try:
        for name in sorted(os.listdir('/content/drive/MyDrive'))[:120]:
            print(' -', os.path.join('/content/drive/MyDrive', name))
    except Exception as exc:
        print('Could not list MyDrive:', repr(exc))

    raise FileNotFoundError(
        'Set BASE to the exact folder path or move arxiv_amostra_1500_abstracts_limitados.json to MyDrive or MyDrive/pipelines.'
    )

PIPE = os.path.dirname(CAMINHO_IN)

SAIDA_JSONL = os.path.join(PIPE, 'arxiv_amostra_1500_com_embeddings_atualizada.json')
SAIDA_NPY   = os.path.join(PIPE, 'arxiv_amostra_1500_embeddings_atualizada.npy')
SAIDA_IDS   = os.path.join(PIPE, 'arxiv_amostra_1500_ids_atualizada.csv')

print('Input file:', CAMINHO_IN)
print('Output folder:', PIPE)
print('Updated outputs:')
print(' -', SAIDA_JSONL)
print(' -', SAIDA_NPY)
print(' -', SAIDA_IDS)

df = pd.read_json(CAMINHO_IN, lines=True)

if CAMPO_TEXTO not in df.columns:
    raise KeyError(f"Required column '{CAMPO_TEXTO}' does not exist. Run Pipeline 2 before Pipeline 3.")

print('Loaded:', df.shape)
print('Columns:', list(df.columns))
df[['id', 'title', 'assigned_category', CAMPO_TEXTO]].head()


## 3. Carregar o modelo e montar os textos de entrada

In [ ]:
from sentence_transformers import SentenceTransformer

assert not INCLUIR_ID, 'Nao use id como feature: campo interno pode poluir/overfittar.'
assert not INCLUIR_CATEGORIA, 'Nao use categoria/subgenero como feature: isso vazaria o rotulo.'

model = SentenceTransformer(MODELO, device=DEVICE)
sep = model.tokenizer.sep_token or '[SEP]'

titulos = df['title'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
abstracts = df[CAMPO_TEXTO].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

if USAR_TITULO:
    textos = [f'{t} {sep} {a}'.strip() for t, a in zip(titulos, abstracts)]
else:
    textos = abstracts.tolist()

print('Exemplo de entrada do modelo, sem id/categoria:\n', textos[0][:500])


## 4. Gerar os embeddings

In [ ]:
import numpy as np

emb = model.encode(
    textos,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # vetores unitarios -> produto interno = similaridade do cosseno
)
print('Shape dos embeddings:', emb.shape)   # (1500, 768)

## 5. Sanidade - os embeddings capturam as categorias?

Se os embeddings forem bons, papers da **mesma** categoria devem ser mais parecidos (cosseno maior) do que papers de categorias **diferentes**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

S = cosine_similarity(emb)
cats = df['assigned_category'].to_numpy(dtype=object)   # ndarray p/ broadcasting (pandas 3.0)
mesma = cats[:, None] == cats[None, :]
tri = np.triu(np.ones_like(S, dtype=bool), k=1)   # ignora diagonal e duplicatas

intra = S[tri & mesma].mean()
inter = S[tri & ~mesma].mean()
print(f'Cosseno medio - mesma categoria:     {intra:.3f}')
print(f'Cosseno medio - categorias diferentes: {inter:.3f}')
print(f'Separacao (intra - inter):           {intra - inter:.3f}  (quanto maior, melhor)')

## 6. Salvar (JSON Lines + .npy) e baixar para o PC

In [ ]:
import numpy as np

# (a) .npy  -> matriz de embeddings (pratico p/ treinar SVM etc.)
np.save(SAIDA_NPY, emb)
df[['id', 'assigned_category']].to_csv(SAIDA_IDS, index=False)   # ordem das linhas = ordem do .npy

# (b) JSON Lines -> tudo junto, com a coluna 'embedding' (mesmo padrao das pipelines anteriores)
df_out = df.copy()
df_out['embedding'] = emb.tolist()
df_out.to_json(SAIDA_JSONL, orient='records', lines=True, force_ascii=False)

print('Salvos no Drive:')
print(' -', SAIDA_NPY)
print(' -', SAIDA_IDS)
print(' -', SAIDA_JSONL)

try:
    from google.colab import files
    files.download(SAIDA_NPY)
    files.download(SAIDA_JSONL)
except Exception as e:
    print('Download automatico indisponivel (rodando fora do Colab?):', e)